# 04 — Fuzzy Matching

Exact matching only finds pairs where values are equal. **Fuzzy** matching finds pairs where a single string column is *similar* (e.g. "Jon" vs "John", or small typos). Matcher returns a **confidence** score (0–1) per pair and you choose a threshold. Here we use the **fuzzy_evaluation** set: 30 known pairs where 15 are identical (exact email finds them) and 15 have name variants on the right (different emails, so only fuzzy on name finds them). You'll see exact miss some pairs and fuzzy recover them, then tune the threshold with `find_best_threshold`.

**Back:** [03 The Measurement Loop](03_measurement_loop.ipynb) · **Next:** [05 Blocking](05_blocking.ipynb)

## 1. Load data and add a fuzzy field

We use the **fuzzy_evaluation** set (50 left, 50 right, 30 known pairs): 15 pairs are identical; 15 pairs are same person but with name variants on the right (e.g. "Katherine" vs "Catherine") and different emails, so only fuzzy on name can find them. Add `full_name` so we have one string field to score.

In [ ]:
import sys
from pathlib import Path
import polars as pl
from matcher import Matcher, find_best_threshold

_tutorial = Path.cwd() / "docs" / "tutorial"
if not (_tutorial / "tutorial_data").exists():
    _tutorial = Path.cwd() if (Path.cwd() / "tutorial_data").exists() else Path.cwd().parent.parent / "docs" / "tutorial"
sys.path.insert(0, str(_tutorial))
from tutorial_data import load_fuzzy_evaluation

left, right, ground_truth = load_fuzzy_evaluation()
left = left.with_columns(
    (pl.col("first_name").fill_null("") + " " + pl.col("last_name").fill_null("")).str.strip_chars().alias("full_name")
)
right = right.with_columns(
    (pl.col("first_name").fill_null("") + " " + pl.col("last_name").fill_null("")).str.strip_chars().alias("full_name")
)
matcher = Matcher(left=left, right=right, left_id="id", right_id="id")
print(left.select(["first_name", "last_name", "full_name"]).head(5))

## 2. Fuzzy as a single rule

`match_fuzzy(field=..., threshold=...)` runs fuzzy matching on that column and returns pairs above the threshold, with a `confidence` column (0–1). One field, one threshold—easy to reason about and to tune.

In [ ]:
results_fuzzy = matcher.match_fuzzy(field="full_name", threshold=0.85)
print(f"Fuzzy matches (full_name, threshold=0.85): {results_fuzzy.count}")
print(results_fuzzy.matches.select(["id", "id_right", "confidence", "full_name", "full_name_right"]).head(5))

## 3. Tune the threshold with find_best_threshold

Picking a threshold by hand is guesswork. Better: run fuzzy with a *low* threshold so you get many scored pairs, then use `find_best_threshold` to sweep and find the cutoff that maximizes F1 on your ground truth. On this data, exact match on email finds only 15 pairs (the name-variant pairs have different emails); fuzzy on full_name can find all 30.

In [ ]:
fuzzy_low = matcher.match_fuzzy(field="full_name", threshold=0.5)
best = find_best_threshold(fuzzy_low.matches, ground_truth, left_id_col="id", right_id_col="id_right")
print(f"Best threshold: {best['best_threshold']}, F1: {best['best_f1']:.2%}")

You now have **exact** (02), the **measurement loop** (03), and **fuzzy** (04). Next we add **blocking** in [05 Blocking](05_blocking.ipynb), then put everything together in [06 Design Your Algorithm](06_design_algorithm.ipynb).